# Debt Payoff Strategies

This notebook demonstrates two popular debt payoff strategies using the Sinking Fund system: the Debt Snowball (smallest first) and Debt Avalanche (highest interest first). We'll use sorted allocation to prioritize debts and track progress toward debt freedom.

**Topics covered:**
- Debt snowball (smallest first) and avalanche (highest interest first)
- Using sorted allocation for debt prioritization
- Tracking progress toward debt freedom


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund import SinkingFund


## Setting Up Debts as Bills

We'll treat each debt as a one-time bill with a due date. This allows us to use the allocation and scheduling system to prioritize payments.


In [ ]:
# Define debts with amounts and due dates.
# Note: Interest rates are stored in service name for reference.
debts = [
    {
        'bill_id': 'credit_card_1',
        'service': 'Credit Card (24% APR)',
        'amount_due': 2500.00,
        'recurring': False,
        'due_date': date(2025, 3, 1)
    },
    {
        'bill_id': 'car_loan',
        'service': 'Car Loan (6% APR)',
        'amount_due': 8500.00,
        'recurring': False,
        'due_date': date(2025, 6, 1)
    },
    {
        'bill_id': 'credit_card_2',
        'service': 'Store Credit Card (18% APR)',
        'amount_due': 1200.00,
        'recurring': False,
        'due_date': date(2025, 4, 1)
    },
    {
        'bill_id': 'personal_loan',
        'service': 'Personal Loan (12% APR)',
        'amount_due': 5000.00,
        'recurring': False,
        'due_date': date(2025, 5, 1)
    }
]

total_debt = sum(debt['amount_due'] for debt in debts)
print(f"Total debt: ${total_debt:,.2f}\n")
print("Debts to pay off:")
for debt in debts:
    print(f"  {debt['service']}: ${debt['amount_due']:,.2f}")


## Strategy 1: Debt Snowball (Smallest First)

The debt snowball method prioritizes paying off the smallest debts first, providing psychological wins and momentum. We'll use sorted allocation with amount-based sorting.


In [ ]:
# Create fund for debt snowball strategy.
snowball_fund = SinkingFund(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 12, 31),
    balance=0.00
)

snowball_fund.create_bills(debts)

# Use sorted allocation with amount-based sorting (smallest first).
snowball_fund.set_allocation_strategy(
    strategy="sorted",
    sort_key="debt_snowball",
    reverse=False  # Smallest first
)

# Generate plan with monthly payments.
snowball_report = snowball_fund.quick_report(
    contribution_interval=30,
    allocation_config={'strategy': 'sorted', 'sort_key': 'debt_snowball', 'reverse': False}
)

print("=== Debt Snowball Strategy (Smallest First) ===\n")

# Show allocation order.
print("Payoff order (by amount, smallest first):")
sorted_debts = sorted(debts, key=lambda d: d['amount_due'])
for i, debt in enumerate(sorted_debts, 1):
    print(f"  {i}. {debt['service']}: ${debt['amount_due']:,.2f}")

# Calculate total contributions needed.
total_contribs = sum(float(data['contributions']['total']) for data in snowball_report.values())
print(f"\nTotal monthly contributions: ${total_contribs:,.2f}")


## Strategy 2: Debt Avalanche (Highest Interest First)

The debt avalanche method prioritizes debts with the highest interest rates, minimizing total interest paid. We'll create a custom allocation that prioritizes by interest rate.

**Note:** For this example, we'll manually sort by interest rate since the built-in allocators use amount or due date. In practice, you could create a custom sort key function.


In [ ]:
# Create fund for debt avalanche strategy.
# Reorder debts by interest rate (highest first).
# Interest rates: Credit Card 1 (24%), Store Card (18%), Personal Loan (12%), Car Loan (6%)
avalanche_debts = [
    debts[0],  # Credit Card 1 (24%)
    debts[2],  # Store Card (18%)
    debts[3],  # Personal Loan (12%)
    debts[1],  # Car Loan (6%)
]

avalanche_fund = SinkingFund(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 12, 31),
    balance=0.00
)

avalanche_fund.create_bills(avalanche_debts)

# Use cascade (due date) allocation, but debts are already ordered by interest rate.
avalanche_report = avalanche_fund.quick_report(
    contribution_interval=30,
    allocation_config={'strategy': 'sorted', 'sort_key': 'cascade'}
)

print("=== Debt Avalanche Strategy (Highest Interest First) ===\n")

print("Payoff order (by interest rate, highest first):")
print("  1. Credit Card (24% APR): $2,500.00")
print("  2. Store Credit Card (18% APR): $1,200.00")
print("  3. Personal Loan (12% APR): $5,000.00")
print("  4. Car Loan (6% APR): $8,500.00")

total_contribs_avalanche = sum(float(data['contributions']['total']) for data in avalanche_report.values())
print(f"\nTotal monthly contributions: ${total_contribs_avalanche:,.2f}")


## Tracking Progress

Let's track how debts are paid off over time with the snowball strategy.


In [ ]:
# Track debt payoff progress month by month.
print("=== Debt Payoff Progress (Snowball Strategy) ===\n")
print("Month    | Account Balance | Debt Remaining | Progress")
print("-" * 65)

check_dates = [
    date(2025, 1, 1),
    date(2025, 3, 1),
    date(2025, 6, 1),
    date(2025, 9, 1),
    date(2025, 12, 31)
]

for check_date in check_dates:
    if check_date in snowball_report:
        account_balance = snowball_report[check_date]['account_balance']['total']
        
        # Calculate remaining debt (total minus account balance).
        debt_remaining = max(0, total_debt - float(account_balance))
        progress_pct = (1 - debt_remaining / total_debt) * 100
        
        month_str = check_date.strftime('%Y-%m')
        print(f"{month_str} | ${float(account_balance):>14,.2f} | ${debt_remaining:>14,.2f} | {progress_pct:>6.1f}%")


## Comparing Strategies

Both strategies have their merits. The snowball method provides psychological momentum, while the avalanche method saves money on interest.


In [ ]:
print("=== Strategy Comparison ===\n")

print("Debt Snowball (Smallest First):")
print("  ✓ Psychological wins from quick payoffs")
print("  ✓ Builds momentum and motivation")
print("  ✗ May pay more interest over time")
print("\nDebt Avalanche (Highest Interest First):")
print("  ✓ Minimizes total interest paid")
print("  ✓ Mathematically optimal")
print("  ✗ May take longer to see first payoff")
print("\nBoth strategies use sorted allocation to prioritize debts.")
print("Choose based on your motivation style and financial goals.")


## Summary

**Key Takeaways:**

1. **Debt Snowball**: Pay smallest debts first using `sort_key="debt_snowball"` with `reverse=False`
2. **Debt Avalanche**: Pay highest interest debts first (requires custom sorting or manual ordering)
3. **Sorted Allocation**: Perfect for debt prioritization strategies
4. **Progress Tracking**: Use reports to monitor debt payoff progress over time

**Implementation Tips:**

- Treat each debt as a one-time bill with its payoff target as the amount due
- Use allocation strategies to control payoff priority
- Adjust contribution intervals to match your payment schedule
- Track progress regularly to stay motivated

**Next Steps:**
- Adjust strategies based on your specific debts and interest rates
- Combine strategies (e.g., pay minimums on all, extra to priority debt)
- Use the system to plan debt payoff timelines
